# 🚦 Traffic Sign Detection — EDA & Training Notebook

This notebook walks through:
1. Dataset exploration (GTSRB)
2. Preprocessing & augmentation
3. Model training
4. Evaluation & visualisation
5. Running inference on sample images


In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
import tensorflow as tf
from collections import Counter
from pathlib import Path

from class_names import CLASS_NAMES, NUM_CLASSES
from data_loader import load_gtsrb_data, preprocess
from model import build_model, compile_model

print('TensorFlow version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 1. Load Data

In [ ]:
DATA_DIR = '../data/'
images, labels = load_gtsrb_data(DATA_DIR)
print(f'Total images : {len(images)}')
print(f'Image shape  : {images[0].shape}')
print(f'Num classes  : {NUM_CLASSES}')

## 2. Class Distribution

In [ ]:
counts = Counter(labels)
sorted_counts = sorted(counts.items())

fig, ax = plt.subplots(figsize=(18, 5))
ax.bar([CLASS_NAMES[k] for k, _ in sorted_counts],
       [v for _, v in sorted_counts],
       color='steelblue')
ax.set_title('Class Distribution in GTSRB Dataset', fontsize=14)
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
plt.show()
print(f'Min samples per class: {min(counts.values())}')
print(f'Max samples per class: {max(counts.values())}')

## 3. Sample Images per Class

In [ ]:
n_cols = 10
n_rows = 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 11))
axes = axes.flatten()

for class_id in range(n_rows * n_cols):
    idx = np.where(labels == class_id)[0][0]
    axes[class_id].imshow(images[idx])
    axes[class_id].set_title(f'{class_id}', fontsize=7)
    axes[class_id].axis('off')

plt.suptitle('One sample per class (first 50)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 4. Model Architecture

In [ ]:
model = build_model(num_classes=NUM_CLASSES)
model.summary()

## 5. Train Model

> ⚠️ This trains from scratch. Skip if you already have `models/traffic_sign_model.keras`.

In [ ]:
# Uncomment to train:
# !python ../src/train.py --data_dir ../data/ --epochs 30 --output_dir ../models/

## 6. Inference on Sample Images

In [ ]:
from predict import load_model, predict_image

model = load_model('../models/traffic_sign_model.keras')

# Pick 10 random images from validation
from data_loader import preprocess
sample_idx = np.random.choice(len(images), 10, replace=False)

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()

for i, idx in enumerate(sample_idx):
    img_rgb = images[idx]
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    class_id, conf, label = predict_image(model, img_bgr)
    true_label = CLASS_NAMES[labels[idx]]
    correct = class_id == labels[idx]
    color = 'green' if correct else 'red'
    axes[i].imshow(img_rgb)
    axes[i].set_title(f'Pred: {label}\n({conf*100:.1f}%)\nTrue: {true_label}',
                      color=color, fontsize=7)
    axes[i].axis('off')

plt.suptitle('Sample Inference Results', fontsize=14)
plt.tight_layout()
plt.show()